In [ ]:
{
    "cells": [
        {
            "cell_type": "markdown",
            "id": "adc83a35",
            "metadata": {},
            "source": [
                "(measure-tutorial)=\n",
                "\n",
                "# Defining Adaptation Measures with python functions",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "3dca8f82",
            "metadata": {},
            "source": [
                "The `Measure` object is the core component of the **Adaptation** module in CLIMADA. It acts as a wrapper for transformation logic, allowing you to model how specific interventions alter the risk landscape (like building sea walls, retrofitting houses, or implementing early warning systems).\n",
                "\n",
                'The primary purpose of a `Measure` is to modify one or more parts of the "Risk Triplet": **Exposures**, **Impact Functions**, and **Hazard**.\n',
                "\n",
                "`Measure` objects can be regrouped into a set with the `MeasureSet` object.",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "f51842c7",
            "metadata": {},
            "source": ["## Quickstart"],
        },
        {
            "cell_type": "markdown",
            "id": "d369e8bd-1f00-4435-9b35-ef03bba2af97",
            "metadata": {},
            "source": [
                "Assume you want to model two measures as follow:\n",
                "- The first one is a targeted awareness campaign, which effect is modelled as a change of one specific impact function. Costing 500'000.\n",
                "- The second one is both building a sea wall which effect is modelled as a 0.2 meter reduction of the flood height everywhere, and a building retrofit, modelled as an overal reduction of the value at risk of the exposure. Costing 1M.",
            ],
        },
        {
            "cell_type": "code",
            "execution_count": 1,
            "id": "2c48b021",
            "metadata": {},
            "outputs": [],
            "source": [
                "from climada.entity.measures import Measure, CostIncome, MeasureSet\n",
                "\n",
                "\n",
                "# Define the effect with python functions\n",
                "def targeted_campaign(impact_func_set, impf_id, rescale):\n",
                '    """Rescales the impact function with id impf_id"""\n',
                "    impact_func_set.get_func(fun_id=imp_id).mdd *= rescale\n",
                "    return impact_func_set\n",
                "\n",
                "\n",
                "def building_retrofit(exposures, reduction_factor):\n",
                '    """Reduces the value of all assets by a fixed percentage."""\n',
                '    exposures.gdf["value"] *= 1 - reduction_factor\n',
                "    return exposures\n",
                "\n",
                "\n",
                "def sea_wall_protection(hazard, reduction):\n",
                '    """Reduces flood intensity by an absolute amount."""\n',
                "    hazard.intensity.data -= reduction\n",
                "    return hazard\n",
                "\n",
                "\n",
                "# Initialize the Measure objects\n",
                "campaign = Measure(\n",
                '    name="Targeted awareness campaign",\n',
                "    impfset_changes=targeted_campaign,\n",
                "    cost_income=CostIncome(init_cost=500_000),\n",
                ")\n",
                "\n",
                "retrofitting_and_seawall = Measure(\n",
                '    name="Building Retrofit and seawall construction",\n',
                "    exposures_changes=building_retrofit,\n",
                "    hazard_changes=sea_wall_protection,\n",
                "    cost_income=CostIncome(init_cost=1_000_000),\n",
                ")",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "02cd2812",
            "metadata": {},
            "source": [
                "The measure can then be used either to create a new triplet corresponding to the situation with measure in effect\n",
                "or to compute the impact directly (both lines are independent):\n",
                "\n",
                "```python\n",
                "new_exp, new_impf, new_haz = retrofitting_and_seawall.apply(exp, impf, haz, kwargs_exposure={'reduction_factor': 0.1}, kwargs_hazard={'reduction': 0.3})\n",
                "\n",
                "# or directly:\n",
                "impact_with_campaign_active = campaign.calc_impact(exp, impf, haz, kwargs_impfset={'rescale': 0.2})\n",
                "```\n",
                "\n",
                "Much like impact functions, the measure can also be regrouped into a set with `MeasureSet`:\n",
                "\n",
                "```python\n",
                "\n",
                "meas_set = MeasureSet([campaign, retrofitting_and_seawall])\n",
                "```",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "6f4650c6",
            "metadata": {},
            "source": [
                "## Core logic of the Measure object\n",
                "\n",
                "When you initialize a `Measure`, you can provide one to three callables (python functions):\n",
                "\n",
                "1. **`exposures_changes`**: Modifies the `Exposures` object, the asset location, value, or metadata (e.g., relocating houses from a flood zone).\n",
                "2. **`impfset_changes`**: Modifies the `ImpactFuncSet` object, the vulnerability of assets (e.g., updating an impact function because houses were elevated).\n",
                "3. **`hazard_changes`**: Modifies the `Hazard` object, the intensity or frequency of events (e.g., a dam reducing flood depths).\n",
                "\n",
                "```{note}\n",
                'By default, changes are the `identity_function`, so if you do not provide a function e.g., "exposures_changes" the measure once applied will return the exposure unchanged.\n',
                "```\n",
                "\n",
                'The **only** requirement for these python functions is to **take** and **return** an `Exposures` object (or `ImpactFuncSet` or `Hazard` objects respectively). This enables high modeling flexibility where changes from an adaptation option can be modelled from a simple "loading another file for the hazard or exposure", to an advanced function.\n',
                "\n",
                "One thing to note, is that the `apply()` method of a measure always passes the entire triplet (`base_exposures`, `base_impfset`, `base_hazard`, i.e., the risk triplet before the measure is applied) to every transformation function. This means you can easily write complex logic that relie on any of these component. For example:\n",
                "\n",
                '> "Only reduce the exposure value (Exposure change) in areas where the peak water depth (Hazard context) exceeds 2 meters."\n',
                "\n",
                "Could be defined like this:",
            ],
        },
        {
            "cell_type": "code",
            "execution_count": 2,
            "id": "7cd53344",
            "metadata": {},
            "outputs": [],
            "source": [
                "import numpy as np\n",
                "\n",
                "\n",
                "def reduce_exposure_by_depth(exposures, reduction_factor=0.2, threshold=2.0, **kwargs):\n",
                '    """\n',
                "    Reduces asset value by a percentage, but only for assets located where\n",
                "    the peak hazard intensity (e.g., water depth) exceeds the threshold.\n",
                '    """\n',
                "    # Retrieve the hazard context passed by Measure.apply\n",
                '    base_hazard = kwargs.get("base_hazard")\n',
                "\n",
                "    # Get the peak intensity for each centroid across all events\n",
                "    peak_depths = np.array(base_hazard.intensity.max(axis=0).todense()).flatten()\n",
                "\n",
                "    # Ensure exposures are mapped to hazard centroids\n",
                '    if "centr_ID" not in exposures.gdf.columns:\n',
                "        exposures.assign_centroids(base_hazard)\n",
                "\n",
                "    # Identify which exposure points meet the depth criteria\n",
                "    # 'centr_ID' maps each exposure row to a column index in\n",
                "    # the hazard intensity matrix\n",
                '    centr_indices = exposures.gdf["centr_ID"].values\n',
                "    point_peak_depths = peak_depths[centr_indices]\n",
                "\n",
                "    # We apply the reduction only where Depth > threshold\n",
                "    mask = point_peak_depths > threshold\n",
                '    exposures.gdf.loc[mask, "value"] *= 1 - reduction_factor\n',
                "\n",
                "    return exposures",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "10a5ad67",
            "metadata": {},
            "source": [
                "```python\n",
                "# --- Usage Example ---\n",
                "# Define the measure\n",
                "flood_protection = Measure(\n",
                '    name="High-Threshold Value Protection",\n',
                "    exposures_changes=reduce_exposure_by_depth\n",
                ")\n",
                "\n",
                "# When you call apply, the `Hazard` haz is automatically passed to the function\n",
                "# as 'base_hazard'\n",
                "new_exp, _, _ = flood_protection.apply(\n",
                "    exp, impf, haz,\n",
                "    kwargs_exposures={'reduction_factor': 0.5, 'threshold': 2.0}\n",
                ")\n",
                "\n",
                '# You can also overide this with another "base" hazard in the\n',
                "# `kwarg_exposures` if you want.\n",
                "#\n",
                "# This could be for instance if the flood protection is\n",
                "# now defined not for the full probabilistic hazard set\n",
                "# but for a specific event, or a subset.\n",
                "new_exp, _, _ = flood_protection.apply(\n",
                "    exp, impf, haz,\n",
                "    kwargs_exposures={'base_hazard'= other_hazard,\n",
                "    'reduction_factor': 0.5, 'threshold': 2.0}\n",
                ")\n",
                "```",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "8baaf95c",
            "metadata": {},
            "source": [
                "```{note}\n",
                "The functions you define yourself are automatically made to accept keyword arguments if they do not already.\n",
                "```\n",
                "\n",
                "```{note}\n",
                "The functions do not necessarily need to use the Exposures|Hazard|ImpactFuncSet object, but they still have to accept it as an argument.\n",
                "```",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "a984d0a7",
            "metadata": {},
            "source": [
                "### Deep copies\n",
                "\n",
                'To prevent accidental "in-place" modification of your baseline data, `Measure` utilizes `copy.deepcopy()`.\n',
                "If `enforce_copy=True` in `apply()` (the default case), the object is cloned before the transformation function runs, ensuring the original objects remains untouched.",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "6d58543f",
            "metadata": {},
            "source": [
                "### Measure cash flow (`CostIncome`)\n",
                "\n",
                "The `cost_income` attribute allows the measure to carry its own. It can notably store:\n",
                "\n",
                "* Initial investment costs.\n",
                "* Maintenance costs over time.\n",
                "* Potential indirect income generated by the measure.\n",
                "\n",
                "For more information and advanced usage see the [detailed tutorial on cost income object](cost-income-tutorial).",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "b5c1873f",
            "metadata": {},
            "source": [
                "## Defining measures using measure configuration\n",
                "\n",
                "Defining adaptation measures with Python functions can be difficult if you are not familiar with Python. For common changes, a `Measure` can instead be described *declaratively* with a `MeasureConfig` object. A configuration can also be written to, and read from, a YAML file (see the [detailed tutorial](measure-config-tutorial)).\n",
                "\n",
                "A `MeasureConfig` aggregates one configuration object per component of the measure:\n",
                "\n",
                "| Field | Class | Main parameters |\n",
                "|---|---|---|\n",
                "| `hazard_modifier` | `HazardModifierConfig` | `haz_int_mult`, `haz_int_add`, `impact_rp_cutoff`, `new_hazard_path` |\n",
                "| `impfset_modifier` | `ImpfsetModifierConfig` | `impf_ids`, `impf_{mdd,paa,int}_{mult,add}`, `new_impfset_path` |\n",
                "| `exposures_modifier` | `ExposuresModifierConfig` | `reassign_impf_id`, `new_exposures_path` |\n",
                "| `cost_income` | `CostIncomeConfig` | same parameters as `CostIncome` |\n",
                "\n",
                'All parameters default to "no change" (multipliers of `1.0`, offsets of `0.0`, `None` otherwise). A `new_*_path` loads a replacement object from file; if modifiers are also given, they are applied after the replacement (and a warning is issued).',
            ],
        },
        {
            "cell_type": "markdown",
            "id": "4e7a2c91",
            "metadata": {},
            "source": [
                "### Creating a measure from a configuration\n",
                "\n",
                "The measure below reduces the mean damage degree of impact function `1` by 25 %, and removes the events that are more frequent than a 1-in-20-year event (ranked by their **impact** in the baseline situation not their intensity).\n",
                "`Measure.from_config` converts the configuration into the corresponding transformation functions.",
            ],
        },
        {
            "cell_type": "code",
            "execution_count": 3,
            "id": "c3d90b57",
            "metadata": {},
            "outputs": [],
            "source": [
                "from climada.entity.measures.measure_config import (\n",
                "    CostIncomeConfig,\n",
                "    ExposuresModifierConfig,\n",
                "    HazardModifierConfig,\n",
                "    ImpfsetModifierConfig,\n",
                "    MeasureConfig,\n",
                ")\n",
                "\n",
                "config = MeasureConfig(\n",
                '    name="Flood proofing and protection standard",\n',
                '    haz_type="FL",\n',
                "    impfset_modifier=ImpfsetModifierConfig(\n",
                '        haz_type="FL", impf_ids=1, impf_mdd_mult=0.75\n',
                "    ),\n",
                '    hazard_modifier=HazardModifierConfig(haz_type="FL", impact_rp_cutoff=20),\n',
                "    exposures_modifier=ExposuresModifierConfig(),  # no change, note that you can also just leave it out entirely\n",
                "    cost_income=CostIncomeConfig(init_cost=1_000_000, periodic_cost=50_000),\n",
                "    color_rgb=(0.1, 0.4, 0.8),\n",
                ")\n",
                "\n",
                "measure = Measure.from_config(config)",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "a81f6d02",
            "metadata": {},
            "source": [
                "The resulting `Measure` is used like any other. The `base_*` triplet needed by `impact_rp_cutoff` is provided automatically by `Measure.apply`.\n",
                "\n",
                "```python\n",
                "imp_with_measure = measure.calc_impact(exp, impfset, haz)\n",
                "```",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "824e1dd3",
            "metadata": {},
            "source": ["## Advanced examples of measures"],
        },
        {
            "cell_type": "markdown",
            "id": "f1a15910-6de8-416f-982f-dd0e53fc13cf",
            "metadata": {},
            "source": [
                "Measures are often modelled as a fixed change. In practice, the effect of an adaptation action often depends on **when** it is evaluated (for instance, a relocation programme that is phased in over several years) or on **additional parameters** (regions targeted, share of value removed, protection threshold).\n",
                "\n",
                "The examples below illustrate three patterns:\n",
                "\n",
                "1. **Exposures**: a linear phase-in of asset relocation (`year`, `start_year`, `full_year`, `fraction`, `regions`).\n",
                "2. **Impact functions**: a year-dependent reduction of the mean damage ratio.\n",
                "3. **Hazard**: a step-function protection standard that needs the baseline impacts, and therefore the `base_*` triplet.",
            ],
        },
        {
            "cell_type": "code",
            "execution_count": 4,
            "id": "7e17dfb9-b760-48df-8878-be1d547290a3",
            "metadata": {},
            "outputs": [],
            "source": [
                "def phase_in(year, start_year, full_year):\n",
                '    """Linear effectiveness ramp. (Helper function)\n',
                "\n",
                "    Parameters\n",
                "    ----------\n",
                "    year : int\n",
                "        Year at which the measure is evaluated.\n",
                "    start_year : int\n",
                "        Year at which the measure starts having an effect.\n",
                "    full_year : int\n",
                "        Year at which the measure is fully effective.\n",
                "\n",
                "    Returns\n",
                "    -------\n",
                "    float\n",
                "        Effectiveness in [0, 1].\n",
                '    """\n',
                "    return float(np.clip((year - start_year) / (full_year - start_year), 0, 1))\n",
                "\n",
                "\n",
                "def relocate_assets(exposures, *, year, start_year, full_year, fraction, regions):\n",
                '    """Progressively remove asset value from exposed regions.\n',
                "\n",
                "    Parameters\n",
                "    ----------\n",
                "    exposures : climada.entity.exposures.base.Exposures\n",
                "        Exposures to modify (already a copy if ``enforce_copy=True``).\n",
                "    year : int\n",
                "        Evaluation year.\n",
                "    start_year, full_year : int\n",
                "        Start and end of the linear phase-in.\n",
                "    fraction : float\n",
                "        Share of value removed once the measure is fully implemented.\n",
                "    regions : list of int\n",
                "        ``region_id`` values targeted by the measure.\n",
                "\n",
                "    Returns\n",
                "    -------\n",
                "    climada.entity.exposures.base.Exposures\n",
                "        Exposures with reduced value in the targeted regions.\n",
                '    """\n',
                "    ramp = phase_in(year, start_year, full_year)\n",
                '    mask = exposures.gdf["region_id"].isin(regions)\n',
                '    exposures.gdf.loc[mask, "value"] *= 1 - fraction * ramp\n',
                "    return exposures\n",
                "\n",
                "\n",
                "def flood_proofing(\n",
                '    impfset, *, year, start_year, full_year, reduction, haz_type="FL", impf_id=1\n',
                "):\n",
                '    """Scale down the mean damage ratio of one impact function.\n',
                "\n",
                "    Parameters\n",
                "    ----------\n",
                "    impfset : climada.entity.impact_funcs.impact_func_set.ImpactFuncSet\n",
                "        Impact function set to modify.\n",
                "    year : int\n",
                "        Evaluation year.\n",
                "    start_year, full_year : int\n",
                "        Start and end of the linear phase-in.\n",
                "    reduction : float\n",
                "        Relative MDD reduction once fully effective.\n",
                "    haz_type : str, optional\n",
                "        Hazard type of the targeted function.\n",
                "    impf_id : int, optional\n",
                "        Id of the targeted function.\n",
                "\n",
                "    Returns\n",
                "    -------\n",
                "    climada.entity.impact_funcs.impact_func_set.ImpactFuncSet\n",
                "        Impact function set with the modified function.\n",
                '    """\n',
                "    ramp = phase_in(year, start_year, full_year)\n",
                "    impf = impfset.get_func(haz_type=haz_type, fun_id=impf_id)\n",
                "    impf.mdd = impf.mdd * (1 - reduction * ramp)\n",
                "    return impfset\n",
                "\n",
                "\n",
                "def protection_standard(\n",
                "    hazard, *, year, start_year, threshold, base_exposures, base_impfset, base_hazard\n",
                "):\n",
                '    """Remove events whose baseline impact is below a protection threshold.\n',
                "\n",
                "    Parameters\n",
                "    ----------\n",
                "    hazard : climada.hazard.base.Hazard\n",
                "        Hazard to modify.\n",
                "    year : int\n",
                "        Evaluation year.\n",
                "    start_year : int\n",
                "        Year the protection is in place (no effect before).\n",
                "    threshold : float\n",
                "        Events with baseline total impact below this value are removed.\n",
                "    base_exposures, base_impfset, base_hazard\n",
                "        Unmodified triplet, provided by ``Measure.apply``.\n",
                "\n",
                "    Returns\n",
                "    -------\n",
                "    climada.hazard.base.Hazard\n",
                "        Hazard without the protected events. Frequencies of the remaining\n",
                "        events are left unchanged.\n",
                '    """\n',
                "    from climada.engine.impact_calc import ImpactCalc\n",
                "\n",
                "    if year < start_year:\n",
                "        return hazard\n",
                "    imp = ImpactCalc(base_exposures, base_impfset, base_hazard).impact(\n",
                "        assign_centroids=False\n",
                "    )\n",
                "    return hazard.select(event_id=imp.event_id[imp.at_event >= threshold])",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "937ded7c-05be-4708-ad08-8d366e287406",
            "metadata": {},
            "source": [
                "The parameters can then be provided when the measure is applied, or the impact is computed:"
            ],
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "id": "2bcbb4ec-701d-4e38-9321-e86a7dea9538",
            "metadata": {},
            "outputs": [],
            "source": [
                "from climada.entity.measures import Measure\n",
                "\n",
                "meas_relocation = Measure(\n",
                '    "relocation",\n',
                "    exposures_changes=relocate_assets,\n",
                ")\n",
                "\n",
                "meas_relocation_and_standard = Measure(\n",
                '    "relocation_and_standard",\n',
                "    exposures_changes=relocate_assets,\n",
                "    hazard_changes=protection_standard,\n",
                ")",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "4e2ea806-8bca-4681-8543-750b9b303a59",
            "metadata": {},
            "source": [
                "```python\n",
                "exp_relocated, _, _ = meas_relocation.apply(\n",
                "    exp,\n",
                "    impfset,\n",
                "    haz,\n",
                "    kwargs_exposures=dict(\n",
                "        year=2030, start_year=2025, full_year=2045, fraction=0.3, regions=[1, 2]\n",
                "    ),\n",
                ")\n",
                "\n",
                "imp_with_relocation_and_standards = meas_relocation_and_standard.calc_impact(\n",
                "    exp,\n",
                "    impfset,\n",
                "    haz,\n",
                "    kwargs_exposures=dict(\n",
                "        year=2030, start_year=2025, full_year=2045, fraction=0.3, regions=[1, 2]\n",
                "    ),\n",
                "    kwargs_hazard=dict(year=2030, start_year=2025, full_year=2045, threshold=100000),\n",
                ")\n",
                "```",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "fabfd29c-0b5e-442a-912f-3cbfac45f78d",
            "metadata": {},
            "source": [
                "## Using `MeasureSet` objects\n",
                "\n",
                "`MeasureSet` are used to group a set of measure together, notably to compare them in a cost benefit analysis.\n",
                "\n",
                "`MeasureSet` are build from a list of measures, taking the example from the quickstart:",
            ],
        },
        {
            "cell_type": "code",
            "execution_count": 6,
            "id": "04e747b4-f856-43c3-ba43-a1878c112afe",
            "metadata": {},
            "outputs": [],
            "source": [
                "from climada.entity.measures.measure import Measure\n",
                "from climada.entity.measures.measure_set import MeasureSet\n",
                "from climada.entity.measures.cost_income import CostIncome\n",
                "\n",
                "\n",
                "# Define the effect with python functions\n",
                "def targeted_campaign(impact_func_set, impf_id, rescale):\n",
                '    """Rescales the impact function with id impf_id"""\n',
                "    impact_func_set.get_func(fun_id=imp_id).mdd *= rescale\n",
                "    return impact_func_set\n",
                "\n",
                "\n",
                "def building_retrofit(exposures, reduction_factor):\n",
                '    """Reduces the value of all assets by a fixed percentage."""\n',
                '    exposures.gdf["value"] *= 1 - reduction_factor\n',
                "    return exposures\n",
                "\n",
                "\n",
                "def sea_wall_protection(hazard, reduction):\n",
                '    """Reduces flood intensity by an absolute amount."""\n',
                "    hazard.intensity.data -= reduction\n",
                "    return hazard\n",
                "\n",
                "\n",
                "# Initialize the Measure objects\n",
                "campaign = Measure(\n",
                '    name="Targeted awareness campaign",\n',
                "    impfset_changes=targeted_campaign,\n",
                "    cost_income=CostIncome(init_cost=500_000),\n",
                ")\n",
                "\n",
                "retrofitting_and_seawall = Measure(\n",
                '    name="Building Retrofit and seawall construction",\n',
                "    exposures_changes=building_retrofit,\n",
                "    hazard_changes=sea_wall_protection,\n",
                "    cost_income=CostIncome(init_cost=1_000_000),\n",
                ")\n",
                "\n",
                "meas_set = MeasureSet([campaign, retrofitting_and_seawall])",
            ],
        },
        {
            "cell_type": "markdown",
            "id": "024822ed-53fc-406e-a9f0-012b244113c8",
            "metadata": {},
            "source": [
                "You can list the measures names of a set, get its size, check if it contains a measure (by name), `append()` an additional measure, `select()` them by their name."
            ],
        },
        {
            "cell_type": "code",
            "execution_count": 7,
            "id": "2d532d22-6b74-45a4-8b14-3e7b9ede2def",
            "metadata": {},
            "outputs": [
                {
                    "name": "stdout",
                    "output_type": "stream",
                    "text": [
                        "There are 2 measures in the set:\n",
                        " ['Targeted awareness campaign', 'Building Retrofit and seawall construction']\n",
                        "True\n",
                        "False\n",
                    ],
                }
            ],
            "source": [
                'print(f"There are {len(meas_set)} measures in the set:\\n", meas_set.names)\n',
                'print("Targeted awareness campaign" in meas_set)\n',
                'print("Mangroves" in meas_set)',
            ],
        },
        {
            "cell_type": "code",
            "execution_count": 8,
            "id": "cdfc53b8-5da4-4b72-8fa5-7a7f5e18a6ea",
            "metadata": {},
            "outputs": [
                {
                    "data": {
                        "text/plain": [
                            "{'Targeted awareness campaign': <climada.entity.measures.measure.Measure at 0x71211d248bd0>}"
                        ]
                    },
                    "execution_count": 8,
                    "metadata": {},
                    "output_type": "execute_result",
                }
            ],
            "source": ['meas_set.select("Targeted awareness campaign")'],
        },
        {
            "cell_type": "markdown",
            "id": "8a29d4b4-983f-4fbd-894b-01716c899dfc",
            "metadata": {},
            "source": [
                "You can also combine or compose measures of a `MeasureSet` into one `Measure`.\n",
                "\n",
                "1. *Combining* (`.combine()`) merges the effects of all the measures, taking the best possible effect when multiple measures affect the same dimension.\n",
                "\n",
                "For instance:\n",
                "- When several measures of the set reduce the intensity of the hazard, the combined effect will be the point-wise highest reduction across all measures.\n",
                "- When several measures modify the same impact function, the combined measure uses the point-wise minimum of all the modified impact functions.\n",
                "\n",
                "2. *Composing* (`.compose()`) does the mathematical composition of the measures, i.e., for a list of measures `[m1, m2, m3]`, the resulting transformation is `m1(m2(m3(x)))`. Note that this means measure `m3` is applied first, and the order of list of measures given to the `MeasureSet` matters!",
            ],
        },
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python [conda env:climada_env_dev]",
            "language": "python",
            "name": "conda-env-climada_env_dev-py",
        },
        "language_info": {
            "codemirror_mode": {"name": "ipython", "version": 3},
            "file_extension": ".py",
            "mimetype": "text/x-python",
            "name": "python",
            "nbconvert_exporter": "python",
            "pygments_lexer": "ipython3",
            "version": "3.11.15",
        },
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}